# RustPower Lesson 4: Direct Pandapower Solver Acceleration ⚡

While `rustpower.PowerGrid` provides a high-level ECS-based interface, you may sometimes want to keep your existing `pandapower` model building code, and only accelerate the bottleneck mathematical calculations (the sparse Newton-Raphson iteration).

RustPower exposes a low-level solver wrapper `rustpower.solver.NewtonSolver` specifically for this purpose. This notebook demonstrates how to extract the system matrices directly from a `pandapower` net and use the high-performance `NewtonSolver` backend to compute the power flow.

In [1]:
import numpy as np
import pandapower as pp
import pandapower.networks as nw
from rustpower.solver import NewtonSolver
import time

## 1. Load a Pandapower Network and Run Native Solve

We load the IEEE 118 case and run the native solver to get the ground-truth voltage solution and initialize the internal matrices (`Ybus`, `Sbus`, etc.).

In [2]:
net = nw.case118()

# Warm up and solve natively
t0 = time.perf_counter()
pp.runpp(net, algorithm='nr', init='flat')
t_pp = (time.perf_counter() - t0) * 1000
print(f"Native Pandapower solved in {t_pp:.2f} ms")

# Extract the exact matrices used in the final iteration
ppci = net["_ppc"]
internal = ppci["internal"]
v_pp = internal["V"]     # Converged voltage vector
Ybus = internal["Ybus"]   # CSR sparse admittance matrix
Sbus = internal["Sbus"]   # Complex bus injection vector

pq = internal["pq"]
pv = internal["pv"]
ref = internal["ref"]

print(f"Ybus shape: {Ybus.shape}, non-zero elements: {Ybus.nnz}")

## 2. Prepare Inputs for RustPower's NewtonSolver

RustPower requires the buses to be partitioned as `[PQ | PV | Slack]`. We construct the permutation vector `p_vec` to map the original indices to this layout, and the inverse permutation `p_inv` to map the solved voltages back. We also build a flat start vector honoring the setpoint magnitudes.

In [3]:
# Permutation vector: PQ buses first, then PV, then Slack/Ref
p_vec = np.concatenate([pq, pv, ref]).astype(np.int64)

# Inverse permutation vector to recover original bus order
p_inv = np.zeros(len(p_vec), dtype=np.int64)
p_inv[p_vec] = np.arange(len(p_vec), dtype=np.int64)

# Flat start: set slack voltage setpoint and PV magnitude setpoints
v_init = np.ones(Ybus.shape[0], dtype=np.complex128)
v_init[pv] = np.abs(v_pp[pv])
v_init[ref] = v_pp[ref]

print(f"Permuted order: {len(pq)} PQ buses, {len(pv)} PV buses, {len(ref)} Slack/Ref buses.")

## 3. Setup and Solve with NewtonSolver

We initialize `NewtonSolver`, pass the sparse matrix indices (`indptr`, `indices`, `data`), target injections `Sbus`, and permutations. 

RustPower's `setup_context` performs the direct $O(NNZ)$ CSR-to-CSC permutation and builds the symbolic Jacobian structure. The `solve()` method runs the Newton-Raphson process on KLU in a branch-free and zero-allocation hot path.

In [4]:
solver = NewtonSolver()

# Setup the solver context
t_start_setup = time.perf_counter()
solver.setup_context(
    y_indptr=Ybus.indptr,
    y_indices=Ybus.indices,
    y_data=Ybus.data,
    s_bus=Sbus,
    v_init=v_init,
    p_vec=p_vec.tolist(),
    p_inv=p_inv.tolist(),
    npv=len(pv),
    npq=len(pq)
)
t_setup = (time.perf_counter() - t_start_setup) * 1000

# Execute the Newton-Raphson solve
t_start_solve = time.perf_counter()
converged = solver.solve()
t_solve = (time.perf_counter() - t_start_solve) * 1000

print(f"Setup Context Time: {t_setup:.2f} ms")
print(f"Pure Solve Time:    {t_solve:.2f} ms")
print(f"Converged:          {converged}")

## 4. Extract and Verify Accuracy

We retrieve the computed voltages and verify that they are mathematically identical to pandapower's converged voltages (up to solver tolerance).

In [5]:
v_rust = solver.get_voltage()

# Calculate the L2 norm of the difference
diff = np.linalg.norm(v_rust - v_pp)
print(f"L2 Norm Difference vs Pandapower: {diff:.2e}")
assert diff < 1e-8, "Solver accuracy validation failed!"